# 🧬 RareGraph Interactive Demo

Runs the pipeline stage-by-stage, **persisting each stage's output to disk**
so you can inspect and diff against the original `rare_dx_mcp` pipeline.

**Prerequisites**
1. `pip install -e .` from the repo root.
2. `python scripts/00_download_hpo.py`
3. Edit `configs/default.yaml` → `paths.kg_path` to your harmonized KG.
4. `python scripts/01_build_indexes.py --config configs/default.yaml`
5. Start Jupyter from the repo root.

In [1]:
import sys, os, json
from pathlib import Path

if 'src' not in sys.path[0]:
    sys.path.insert(0, str(Path('..').resolve() / 'src'))
    os.chdir(Path('..').resolve())
print('Working dir:', Path.cwd())

Working dir: /mnt/isilon/wang_lab/quan/projects/MultiAgentLLM


In [2]:
import logging
import pandas as pd
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)

from raregraph.core.config import load_config
from raregraph.core.logging import setup_logger
from raregraph.core.utils import ensure_dir, write_json
from raregraph.genomics.adapters import discover_genomics_result
from raregraph.orchestration.host import RareGraphHost
from raregraph.pipeline.vision_prefetch import prefetch_vision_for_cases

logger = setup_logger(level=logging.INFO)
cfg = load_config('configs/default.yaml')

print('Config loaded. KG path:', cfg.paths.kg_path)

/home/nguyenqm/miniconda3/envs/rare_dx_mcp/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/nguyenqm/miniconda3/envs/rare_dx_mcp/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Config loaded. KG path: data/knowledge_graph/Mondo_diseases_KG_combined.json


In [3]:
cfg.project.dataset = 'demo'
DATASET = cfg.project.dataset           # 'demo' by default; change via config
INPUT_ROOT = Path('inputs') / DATASET
TEXT_DIR   = INPUT_ROOT / 'text'
HPO_DIR    = INPUT_ROOT / 'free_hpo'
VCF_DIR    = INPUT_ROOT / 'vcf'
IMAGE_DIR  = INPUT_ROOT / 'image'

def _first_existing(*candidates: Path) -> Path | None:
    for c in candidates:
        if c.exists():
            return c
    return None

def _discover_case_ids(input_root: Path) -> set:
    """Scan all modality folders and union the case IDs found."""
    case_ids = set()
    text_dir = input_root / 'text'
    hpo_dir = input_root / 'free_hpo'
    vcf_dir = input_root / 'vcf'
    image_dir = input_root / 'image'

    if text_dir.exists():
        for p in text_dir.glob('*.txt'):
            case_ids.add(p.stem)
    if hpo_dir.exists():
        for p in hpo_dir.glob('*.txt'):
            case_ids.add(p.stem)
    if vcf_dir.exists():
        for p in vcf_dir.glob('*.vcf'):
            case_ids.add(p.stem)
        for p in vcf_dir.glob('*.vcf.gz'):
            case_ids.add(p.stem.replace('.vcf', ''))
    if image_dir.exists():
        for p in image_dir.glob('*'):
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.tiff'):
                case_ids.add(p.stem)
    return case_ids

all_case_ids = sorted(_discover_case_ids(INPUT_ROOT))

if len(all_case_ids) == 0:
    raise RuntimeError(
        f'No patient files found in {INPUT_ROOT}. '
        f'Expected at least one file under text/, hpo/, vcf/, or image/.'
    )

rows = []
for cid in all_case_ids:
    note_path = _first_existing(TEXT_DIR / f'{cid}.txt')
    hpo_path = _first_existing(HPO_DIR / f'{cid}.txt')
    vcf_path = _first_existing(
        VCF_DIR / f'{cid}.vcf',
        VCF_DIR / f'{cid}.vcf.gz',
    )
    img_path = _first_existing(
        IMAGE_DIR / f'{cid}.jpg',
        IMAGE_DIR / f'{cid}.jpeg',
        IMAGE_DIR / f'{cid}.png',
        IMAGE_DIR / f'{cid}.tiff',
    )
    rows.append({
        'case_id': cid,
        'note_path': str(note_path) if note_path else None,
        'hpo_path': str(hpo_path) if hpo_path else None,
        'vcf_path': str(vcf_path) if vcf_path else None,
        'image_path': str(img_path) if img_path else None,
    })

cases_df = pd.DataFrame(rows)
available = cases_df.notna().sum()
print(f'Found {len(cases_df)} cases in {INPUT_ROOT}')
print(f"  with note:  {available.get('note_path', 0)}")
print(f"  with HPO:   {available.get('hpo_path', 0)}")
print(f"  with VCF:   {available.get('vcf_path', 0)}")
print(f"  with image: {available.get('image_path', 0)}")

IDX = 0 # SELECT PATIENT ID HERE
CASE_ID    = cases_df.iloc[IDX]['case_id']
NOTE_PATH  = cases_df.iloc[IDX]['note_path']
HPO_PATH   = cases_df.iloc[IDX]['hpo_path']
VCF_PATH   = cases_df.iloc[IDX]['vcf_path']
IMAGE_PATH = cases_df.iloc[IDX]['image_path']
OUT_DIR    = ensure_dir(f'outputs/{DATASET}/{CASE_ID}')
OUT_DIR = ensure_dir(f'outputs/{DATASET}/{CASE_ID}')
STAGE1_CACHE_PATH = OUT_DIR / 'stage1_text_prefetch.json'
VISION_CACHE_PATH = OUT_DIR / 'stage1_vision_prefetch.json' if IMAGE_PATH else None
GENOMICS_RESULT_PATH = discover_genomics_result(
    CASE_ID,
    VCF_PATH,
    analyzer=getattr(cfg.genomics, 'vcf_analyzer', 'auto'),
    results_dir=getattr(cfg.genomics, 'results_dir', '') or None,
) if VCF_PATH else None
print(f"\033[91m Selected case: {CASE_ID} \033[0m")
print('Stage 1 text cache:', STAGE1_CACHE_PATH)
print('Vision cache:', VISION_CACHE_PATH)
print('Genomics result:', GENOMICS_RESULT_PATH)

Found 1 cases in inputs/demo
  with note:  0
  with HPO:   1
  with VCF:   0
  with image: 0
 Selected case: case001 
Stage 1 text cache: outputs/demo/case001/stage1_text_prefetch.json
Vision cache: None
Genomics result: None


## Stage 0: Load KG and build indexes

In [4]:
vision_mode = getattr(getattr(cfg, 'vision', {}), 'extraction_mode', 'preextract_unload')
if IMAGE_PATH and vision_mode == 'preextract_unload':
    prefetch_vision_for_cases(
        cfg,
        [(CASE_ID, IMAGE_PATH, str(VISION_CACHE_PATH))],
        prompt_dir='configs/prompts',
        overwrite=False,
    )
    print('Vision extraction prefetched before loading text model.')

host = RareGraphHost(cfg)
host.load()
if NOTE_PATH:
    host.precompute_stage1_extractions([
        {
            'case_id': CASE_ID,
            'note_path': NOTE_PATH,
            'stage1_json_path': str(STAGE1_CACHE_PATH),
        }
    ], overwrite=False)
    print('Stage 1 text extraction prefetched/cached.')
print(f'KG loaded: {len(host.kg)} diseases')
print(f'HPO IC stats: p25={host.hpo.ic_p25:.2f}, median={host.hpo.ic_median:.2f}, p75={host.hpo.ic_p75:.2f}')
print(f'Pathognomonic HPOs: {len(host.kg_index.pathognomonic_hpos)}')
print(f'Co-occurrence pairs: {len(host.kg_index.pair_frequency)}')

# Sanity: names should no longer equal IDs (fixed by looking into meta.name)
print('\nSample disease names from KG:')
for did, dname in list(host.kg_index.disease_name.items())[:5]:
    print(f'  {did}  ->  {dname}')

[04/27/26 17:37:36] INFO     === RareGraphHost.load() ===

                    INFO     Loading HPO ontology from data/hpo/hp.obo ...

[04/27/26 17:37:38] INFO     HPO loaded: 19389 terms

                    INFO     Loading KG from data/knowledge_graph/Mondo_diseases_KG_combined.json

[04/27/26 17:37:40] INFO     KG loaded: 15280 diseases

...

(EngineCore pid=3824969) INFO 04-27 17:38:18 [default_loader.py:384] Loading weights took 3.01 seconds
(EngineCore pid=3824969) INFO 04-27 17:38:21 [gpu_model_runner.py:4566] Model loading took 15.27 GiB memory and 4.647042 seconds
(EngineCore pid=3824969) INFO 04-27 17:38:26 [backends.py:988] Using cache directory: /home/nguyenqm/.cache/vllm/torch_compile_cache/728b22bad3/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=3824969) INFO 04-27 17:38:26 [backends.py:1048] Dynamo bytecode transform time: 3.59 s
(EngineCore pid=3824969) INFO 04-27 17:38:28 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.821 s
(EngineCore pid=3824969) INFO 04-27 17:38:28 [monitor.py:48] torch.compile took 5.71 s in total
(EngineCore pid=3824969) INFO 04-27 17:38:28 [decorators.py:296] Directly load AOT compilation from path /home/nguyenqm/.cache/vllm/torch_compile_cache/torch_aot_compile/b83ea48391549cc8f1c1957719325d86b2a08f6393cf45b98e58036e

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 20.16it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 19.73it/s]


(EngineCore pid=3824969) INFO 04-27 17:38:45 [gpu_model_runner.py:5746] Graph capturing finished in 7 secs, took 0.58 GiB
(EngineCore pid=3824969) INFO 04-27 17:38:45 [gpu_worker.py:617] CUDA graph pool memory: 0.58 GiB (actual), 0.59 GiB (estimated), difference: 0.01 GiB (1.3%).
(EngineCore pid=3824969) INFO 04-27 17:38:45 [core.py:281] init engine (profile, create kv cache, warmup model) took 23.99 seconds
INFO 04-27 17:38:47 [llm.py:391] Supported tasks: ['generate']


[04/27/26 17:38:47] INFO     === RareGraphHost.load() complete ===

KG loaded: 15280 diseases
HPO IC stats: p25=6.69, median=8.54, p75=9.63
Pathognomonic HPOs: 5335
Co-occurrence pairs: 2870480

Sample disease names from KG:
  MONDO:0000005  ->  alopecia, isolated
  MONDO:0000009  ->  Rare hemorrhagic disorder due to a constitutional platelet anomaly
  MONDO:0000023  ->  Fever-associated acute infantile liver failure syndrome
  MONDO:0000030  ->  Sleep-related hypermotor epilepsy
  MONDO:0000032  ->  febrile seizures, familial


## Stage 1: Multi-modal extraction

In [ ]:
from raregraph.core.state import PatientCaseState

state = PatientCaseState(
    case_id=CASE_ID,
    note_text=NOTE_PATH,#.read_text(),
    image_path=IMAGE_PATH,
    vcf_path=VCF_PATH,
    free_hpo_path=HPO_PATH,
)
stage1 = host._stage1_extraction(
    state,
    state.note_text,
    IMAGE_PATH,
    HPO_PATH,
    vision_json_path=str(VISION_CACHE_PATH) if VISION_CACHE_PATH else None,
    stage1_json_path=str(STAGE1_CACHE_PATH) if NOTE_PATH else None,
    genomics_result_path=str(GENOMICS_RESULT_PATH) if GENOMICS_RESULT_PATH else None,
)

write_json(stage1, OUT_DIR / 'stage1_extraction.json')
print(f'Saved: {OUT_DIR}/stage1_extraction.json')
stage1

## Stage 2: Normalization + post-processing

In [ ]:
host._stage2_normalization(state)
host._save_patient_evidence(state, OUT_DIR)  # saves stage2_patient_evidence.json + stage2_incongruity.json

print('Normalized HPOs:')
for n in state.normalized_hpo[:15]:
    print(f'  {n.hpo_id:12s}  IC={n.ic:5.2f}  present={n.present}  {n.mention}')
print('\nIncongruity:', state.incongruity.model_dump())
print('Inheritance prior:', state.inheritance_prior)
print(f'\nSaved: stage2_patient_evidence.json, stage2_incongruity.json')

## Stage 3: Candidate retrieval + composite scoring

In [ ]:
pd.set_option('display.max_columns', 40)

In [ ]:
ranked_df, cooc_candidates = host._stage3_retrieval_scoring(state)
ranked_df.to_csv(OUT_DIR / 'stage3_composite_ranking.tsv', sep='\t', index=False)
print(f'{len(ranked_df)} candidates scored. Saved: stage3_composite_ranking.tsv')
from raregraph.core.config import retrieval_retain_top_k
ranked_df = ranked_df.head(retrieval_retain_top_k(host.cfg)).reset_index(drop=True)
ranked_df.head(20)

In [ ]:
ranked_df[ranked_df['disease_id'] == 'MONDO:0007301']

## Stage 4: Frontier consultation (conditional)

In [ ]:
frontier_output, frontier_flags = host._stage4_frontier(state, ranked_df, OUT_DIR)
write_json(frontier_output, OUT_DIR / 'stage4_frontier_output.json')
write_json(frontier_flags, OUT_DIR / 'stage4_frontier_flags.json')

print('Frontier triggered:', frontier_output.get('triggered', False))
print('Reason:', frontier_output.get('trigger_reason', ''))
if frontier_flags:
    print()
    for did, flag in frontier_flags.items():
        print(f"  [{flag['flag_type']:<12}] {did}  lens={flag['lens']}")
        print(f"    {flag['reasoning'][:160]}")

## Stage 5: Audit

In [ ]:
audit_results, ranking_after_audit = host._stage5_audit(state, ranked_df, frontier_flags)

write_json(audit_results, OUT_DIR / 'stage5_audit_results.json')
ranking_after_audit.to_csv(OUT_DIR / 'stage5_ranking_after_audit.tsv', sep='\t', index=False)

import collections
print('Audit distribution:', collections.Counter(a['plausibility'] for a in audit_results))
print('Saved: stage5_audit_results.json, stage5_ranking_after_audit.tsv')
ranking_after_audit.head(20)[['adjusted_rank','disease_name','total_score',
                              'audit_plausibility','audit_multiplier','adjusted_score']]

In [ ]:
ranking_after_audit[ranking_after_audit['disease_id'] == 'MONDO:0007301']

## Stage 6: Adaptive pairwise adjudication

In [ ]:
pairwise = host._stage6_pairwise(state, ranking_after_audit, audit_results, frontier_flags, track='subtype')
write_json(pairwise, OUT_DIR / 'stage6_pairwise_results_subtype.json')
print(f'Pairwise verdicts: {len(pairwise)}. Saved: stage6_pairwise_results_subtype.json')
pd.DataFrame(pairwise).head(15)[['disease_a_name','disease_b_name','prompt_type','winner','strength']]

In [ ]:
# Group-level pairwise (on group IDs)
group_ranked_df = host._aggregate_to_group(ranking_after_audit)
pairwise_group = host._stage6_pairwise(
    state, group_ranked_df, audit_results, frontier_flags, track="group",
) if len(group_ranked_df) > 1 else []
state.pairwise_results_group = pairwise_group
write_json(pairwise_group, OUT_DIR / "stage6_pairwise_results_group.json")

In [ ]:
pd.DataFrame(pairwise_group).head(15)[['disease_a_name','disease_b_name','prompt_type','winner','strength']]

## Stage 7: Rank centrality aggregation

In [ ]:
from raregraph.reasoning.rank_centrality import aggregate_rank

reranked_subtype = aggregate_rank(ranking_after_audit, pairwise, cfg, track_name='subtype')
reranked_subtype.to_csv(OUT_DIR / 'stage7_reranked_subtype.tsv', sep='\t', index=False)
print(f'Saved: stage7_reranked_subtype.tsv')
reranked_subtype.head(15)[['reranked_rank_subtype','disease_name','final_score_subtype','total_score','adjusted_score']]

In [ ]:
reranked_group = aggregate_rank(
    group_ranked_df, pairwise_group, cfg, track_name="group"
) if len(group_ranked_df) > 1 else group_ranked_df
state.reranked_group = reranked_group
reranked_group.to_csv(OUT_DIR / "stage7_reranked_group.tsv", sep="\t", index=False)
print(f'Saved: stage7_reranked_group.tsv')

## Stages 8 + 9

In [ ]:
from raregraph.reasoning.reconciliation import reconcile

from raregraph.reasoning.scorecard import (
    build_scorecard, format_scorecard_text, build_rank_trajectory,
)
# =================== STAGE 8 ===================
reconciled = reconcile(
    reranked_subtype, reranked_group, state, host.kg_index, host.cfg,
    llm=host.text_llm, prompt_dir=host.prompt_dir,
)
state.reconciled = {
    "top_subtype": reconciled.get("top_subtype"),
    "top_group": reconciled.get("top_group"),
    "disagreement": reconciled.get("disagreement", False),
    "tiebreaker": reconciled.get("tiebreaker"),
    "method": reconciled.get("method", ""),
}
write_json(state.reconciled, OUT_DIR / "stage8_reconciled.json")

In [ ]:
# =================== STAGE 9 ===================
final_df_for_scorecard = (
    reconciled["reconciled_df"] if isinstance(reconciled.get("reconciled_df"), pd.DataFrame)
    else reranked_subtype
)
scorecard = build_scorecard(
    state, final_df_for_scorecard, audit_results,
    reconciled, frontier_output,
    top_k=host.cfg.output.scorecard_top_k,
)
state.scorecard = scorecard

if host.cfg.output.save_scorecard_json:
    write_json(scorecard, OUT_DIR / "stage9_scorecard.json")
if host.cfg.output.save_scorecard_txt:
    (OUT_DIR / "stage9_scorecard.txt").write_text(
        format_scorecard_text(scorecard), encoding="utf-8"
    )

### Rank trajectory

In [ ]:
if host.cfg.output.save_rank_trajectory:
    traj = build_rank_trajectory(
        ranked_df, audit_results, reranked_subtype, reconciled, frontier_output
    )
    traj.to_csv(OUT_DIR / "rank_trajectory.tsv", sep="\t", index=False)

In [ ]:
if host.cfg.output.save_rank_trajectory:
    traj = build_rank_trajectory(
        ranked_df, audit_results, reranked_subtype, reconciled, frontier_output
    )
    traj.to_csv(OUT_DIR / "rank_trajectory.tsv", sep="\t", index=False)

In [ ]:
#traj

In [ ]:
#traj[traj['disease_id'] == 'MONDO:0007301']

# Full-pipeline call

Ensures every remaining stage file is persisted to disk.

In [ ]:
# result = host.run_patient(
#     case_id=CASE_ID,
#     note_path=NOTE_PATH,
#     image_path=IMAGE_PATH,
#     vcf_path=VCF_PATH,
#     free_hpo_path=HPO_PATH,
#     vision_json_path=str(VISION_CACHE_PATH) if VISION_CACHE_PATH else None,
#     stage1_json_path=str(STAGE1_CACHE_PATH) if NOTE_PATH else None,
#     genomics_result_path=str(GENOMICS_RESULT_PATH) if GENOMICS_RESULT_PATH else None,
#     output_dir=str(OUT_DIR),
# )

# print('Top 5:')
# for t in result['top']:
#     print(f"  rank {t['reconciled_rank']:3d}  {t['disease_name']}  [audit={t['audit_plausibility']}]  change={t['total_rank_change']:+d}")
# print('\nReconciled:', result['reconciled'])
# print('\nFiles in output dir:')
# for f in sorted(OUT_DIR.iterdir()):
#     print(f'  {f.name}  ({f.stat().st_size} bytes)')